# Assignment #4

### Q1

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import numpy_financial as npf

In [5]:
# Importing excel sheet into python as a dataframe
df_stock_data = pd.read_excel('PriceData_A3.xlsx', sheet_name='Sheet1', index_col=0)
df_stock_data

,NVDA,XOM,PEP,WFC,CMCSA,PFE,T,MS,NKE,FDX
date,,,,,,,,,,
2000-01-01,0.070817,19.086594,18.424389,10.137085,10.241562,14.404564,6.027568,33.158226,4.348876,32.587128
2000-02-01,0.122286,17.344908,17.344580,8.378934,9.499417,12.787477,5.345500,35.365871,2.718046,28.777567
2000-03-01,0.161426,18.064533,18.829321,10.383640,9.796278,14.590761,5.965011,41.610565,3.787344,31.969374
2000-04-01,0.170293,17.992161,19.895611,10.463275,9.261932,16.810558,6.203965,38.535286,4.159306,31.042727
2000-05-01,0.218061,19.294891,22.064798,11.530313,8.920548,17.758324,6.231514,35.982941,4.105442,29.240906
...,...,...,...,...,...,...,...,...,...,...
2024-06-01,123.519287,114.200592,162.403046,58.939735,38.553123,27.590616,18.588745,95.567978,74.743225,296.754791
2024-07-01,117.009987,117.642876,171.365097,58.890114,40.630424,30.114992,18.724928,101.487503,74.527565,300.776337
2024-08-01,119.359795,116.998070,171.573517,58.026714,39.275833,29.010000,19.647734,102.797249,82.949997,297.313293


In [6]:
# Compute simple daily returns as percent change
returns = df_stock_data.pct_change().dropna()  # using a method chain to alter the data table to show %change rather then returns

returns.head()

,NVDA,XOM,PEP,WFC,CMCSA,PFE,T,MS,NKE,FDX
date,,,,,,,,,,
2000-02-01,0.726789,-0.091252,-0.058608,-0.173438,-0.072464,-0.112262,-0.113158,0.066579,-0.375000,-0.116904
2000-03-01,0.320069,0.041489,0.085603,0.239255,0.031250,0.141020,0.115894,0.176574,0.393407,0.110913
2000-04-01,0.054929,-0.004006,0.056629,0.007669,-0.054546,0.152137,0.040059,-0.073906,0.098212,-0.028985
2000-05-01,0.280505,0.072405,0.109028,0.101979,-0.036859,0.056379,0.004441,-0.066234,-0.012950,-0.058043
2000-06-01,0.113913,-0.052649,0.092166,-0.139069,0.034942,0.080760,0.007153,0.164336,-0.071428,0.070423


In [7]:
n_stocks = returns.shape[1]       # using attribute lookup to accessing second collumn in data set
equal_weight = 1 / n_stocks       # 1 divided by the 10 stocks

# Create a Series of weights aligned with the columns of returns
weights = pd.Series(equal_weight, index=returns.columns, name="weight")

weights

NVDA     0.1
XOM      0.1
PEP      0.1
WFC      0.1
CMCSA    0.1
PFE      0.1
T        0.1
MS       0.1
NKE      0.1
FDX      0.1
Name: weight, dtype: float64

In [8]:
cov_matrix = returns.cov() # Pandas Cov method

cov_matrix

,NVDA,XOM,PEP,WFC,CMCSA,PFE,T,MS,NKE,FDX
NVDA,0.029919,0.001914,0.000519,0.000754,0.003168,0.001306,0.001752,0.005712,0.001967,0.004438
XOM,0.001914,0.004003,0.000788,0.001975,0.001401,0.000918,0.001520,0.002199,0.000996,0.001288
PEP,0.000519,0.000788,0.002059,0.001125,0.001008,0.001142,0.001149,0.001026,0.001239,0.001115
WFC,0.000754,0.001975,0.001125,0.006902,0.002312,0.001533,0.001143,0.003280,0.002580,0.002516
CMCSA,0.003168,0.001401,0.001008,0.002312,0.004979,0.001351,0.001627,0.002971,0.001455,0.002509
PFE,0.001306,0.000918,0.001142,0.001533,0.001351,0.003675,0.001178,0.001660,0.001196,0.001177
T,0.001752,0.001520,0.001149,0.001143,0.001627,0.001178,0.004201,0.001265,0.001101,0.001539
MS,0.005712,0.002199,0.001026,0.003280,0.002971,0.001660,0.001265,0.010028,0.002360,0.002937
NKE,0.001967,0.000996,0.001239,0.002580,0.001455,0.001196,0.001101,0.002360,0.006254,0.002455
FDX,0.004438,0.001288,0.001115,0.002516,0.002509,0.001177,0.001539,0.002937,0.002455,0.006551


In [12]:
def portfolio_var_std(returns_df, weights_series, cov_matrix): # Defining a function
    '''  
    Overview:
    Computes portfolio variance and portfolio standard deviation.

    Inputs/Parameters:
    returns_df: DataFrame for stock returns
    weights_series: Series containing portfolio weights
    cov_matrix: Variance/covariance matrix of returns_df

    Output/Result:
    Returns: Portfolio variance and portfolio standard deviation
    ''' 
    # reordering weights_series so the match the order of the cov matrix 
    weights_series = weights_series.loc[cov_matrix.index]   # use .loc to select based on labels

    # Preparing data for port_var calculation
    w = weights_series.values.reshape(-1, 1)   # reshapre values into culumn vectors 
    S = cov_matrix.values                      # covariance matrix

    # Varriance Calculation
    variance = np.dot(np.dot(w.T, S), w).item()

    # Stdev Calculation
    stdev = variance ** 0.5

    return variance, stdev

In [13]:
port_var, port_std = portfolio_var_std(returns, weights, cov_matrix)

port_var, port_std

(0.0023970079453699083, 0.04895924780232953)

In [14]:
port_var_pct = round(port_var * 100, 2)   # variance in percent
port_std_pct = round(port_std * 100, 2)   # std dev in percent

print("Portfolio Variance (%):", port_var_pct)
print("Portfolio Standard Deviation (%):", port_std_pct)

Portfolio Variance (%): 0.24
Portfolio Standard Deviation (%): 4.9


### Q2

In [15]:
def portfolio_cov(returns_df, weights1, weights2, cov_matrix):
    '''
    Overview:
    Computes covariance between two portfolios using Cov = w1' S w2.

    Inputs/Parameters:
    returns_df: DataFrame for stock returns
    weights1: portfolio 1 series weights
    weights2: portfolio 2 series weights 
    cov_matrix: Variance–covariance matrix of returns_df

    Output/Result:
        Covariance between the two portfolios
    '''

    # Align both weight vectors to the order of the covariance matrix
    weights1 = weights1.loc[cov_matrix.index]
    weights2 = weights2.loc[cov_matrix.index]

    # Preparing data using .reshape(-1, 1) puts it into culumn vectors 
    w1 = weights1.values.reshape(-1, 1)
    w2 = weights2.values.reshape(-1, 1)
    S  = cov_matrix.values

    # Compute w1S w2
    cov = np.dot(np.dot(w1.T, S), w2).item()

    return cov

In [16]:
weights_p2 = pd.Series({
    'NVDA': 0.25,
    'XOM' : 0.14,
    'PEP' : 0.65,
    'WFC' : 0.13,
    'CMCSA': -0.04,
    'PFE' : -0.15,
    'T'   : -0.03,
    'MS'  : -0.10,
    'NKE' : 0.20,
    'FDX' : -0.04
}, name='weight_p2')

weights_p2

NVDA     0.25
XOM      0.14
PEP      0.65
WFC      0.13
CMCSA   -0.04
PFE     -0.15
T       -0.03
MS      -0.10
NKE      0.20
FDX     -0.04
Name: weight_p2, dtype: float64

In [17]:
cov_p1_p2 = portfolio_cov(returns, weights, weights_p2, cov_matrix)

# Convert to percent and round to 2 decimals
cov_p1_p2_pct = round(cov_p1_p2 * 100, 2)

print("Covariance between portfolio 1 and 2 (%):", cov_p1_p2_pct)

Covariance between portfolio 1 and 2 (%): 0.22


In [19]:
var_p2, std_p2 = portfolio_var_std(returns, weights_p2, cov_matrix)

var_p2_pct  = round(var_p2 * 100, 2)   # variance in percent
std_p2_pct  = round(std_p2 * 100, 2)   # standard deviation in percent

print("Portfolio 2 Variance (%):", var_p2_pct)
print("Portfolio 2 Standard Deviation (%):", std_p2_pct)

Portfolio 2 Variance (%): 0.34
Portfolio 2 Standard Deviation (%): 5.79


In [20]:
corr_p1_p2 = cov_p1_p2 / (port_std * std_p2)

print("Correlation between portfolio 1 and 2:", round(corr_p1_p2, 4))

Correlation between portfolio 1 and 2: 0.7715


Q2(d) — Interpretation of p

p does not match the envelope portfolio because the weight structure doesn't minimize variance and the correlation of the equal-weight portfolio is not close to +-1. So the portfolio is not on the efficient envelope.